# 18 - Paired audit of alternative-corruption generalization

**CPU is sufficient; Run All on a Python Colab runtime.** Reads completed notebook-17 score/extraction reports, protected notebook-14 summary tables and frozen notebook-12 statistical helpers on Drive. No images, GPU, NBIS installation, training, model inference, new thresholds, checkpoint selection or test evaluation. No need to rerun 02/04/17 after a runtime reset if these Drive reports remain intact.

Notebook 17 showed small FRR/FAR changes on three alternative synthetic mechanisms. This audit quantifies their conditional uncertainty and preserves negative/near-zero results; it does not select a model or tune any corruption.

- Use only the original-input thresholds saved in notebook14, copied into notebook17 and unchanged across methods/conditions. Recompute all forty variants, six databases, all-impression and byte-distinct sensitivity rates; rejected attempts have no score, not score zero.
- Compare input -> M0, input -> L1, input -> L1+Sobel, and matched L1 -> L1+Sobel. Primary summaries: nine-condition equal-weight average, three fixed levels per mechanism, and identity.
- Reuse frozen notebook12 statistical helpers and **2,000 paired probe-finger cluster draws** within each database. All impressions and gallery comparisons of a sampled probe finger stay together. Same draws across methods and conditions, equal database/condition weights. Verify draw hashes against notebook12.
- Pointwise 95% percentile intervals cover attempt FRR/FAR on all impressions. Byte-distinct results are descriptive sensitivity summaries, not separately bootstrapped. Deltas are method minus baseline, in percentage points.
- A contextual table places notebook14 and notebook17 point estimates side by side. **Do not pool the two protocols, subtract their effect sizes as a generalization test, or treat levels as matched difficulty.** They use the same validation identities; they are not independent replications.

**Limitations:** fixed shared galleries, fixed already-calibrated thresholds, selected checkpoints, one training seed, repeated validation development and one deterministic corruption case per source/family/level. Intervals exclude calibration, selection, training-seed and simulator-sampling uncertainty. They are not multiplicity-corrected significance tests, equivalence/non-inferiority tests, real-moisture evidence or held-out-test results. An interval crossing zero does not establish equivalence; lower FRR must be read alongside FAR and original-image harms.

Outputs: `analyses/alternative_corruption_tradeoff_v1/<hash>/`. Completion: `ALTERNATIVE CORRUPTION AUDIT COMPLETE`. Send the overall/family paired intervals and original-image harms before deciding on multi-seed confirmation or a new method. Do not train or open held-out tests from this notebook.


In [ ]:
import hashlib
import inspect
import linecache
import json
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
EVAL_DIR = PROJECT_ROOT / 'evaluations' / 'alternative_corruption_validation_v1' / '476fc096ee2d51f03921'
REFERENCE_DIR = PROJECT_ROOT / 'evaluations' / 'm2_structure_validation_v1' / '1374b21b3b10b3202a8d'
AUDIT12_DIR = PROJECT_ROOT / 'analyses' / 'm1_validation_tradeoff_v1' / '851b89eb4be16b94389a'
METHODS = ('input', 'm0', 'l1_control', 'l1_sobel')
COMPARISONS = [('input', 'm0'), ('input', 'l1_control'), ('input', 'l1_sobel'), ('l1_control', 'l1_sobel')]
POLICIES = ('original_fixed',)
SEED, REPLICATES = 20261110, 2000

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))
print('CPU-only report audit. No images, GPU inference or NBIS subprocesses.')

In [ ]:
completion = json.loads((EVAL_DIR / 'completion.json').read_text(encoding='utf-8'))
configuration = json.loads((EVAL_DIR / 'configuration.json').read_text(encoding='utf-8'))
if (completion['status'] != 'alternative_corruption_validation_complete'
        or completion['configuration_sha256'] != file_digest(EVAL_DIR / 'configuration.json')
        or completion['variants_available'] != 40 or set(completion['methods']) != set(METHODS)):
    raise RuntimeError('Expected completed notebook-17 evaluation with all four methods.')
for name, expected in completion['file_sha256'].items():
    if file_digest(EVAL_DIR / name) != expected:
        raise RuntimeError(f'Completed notebook-17 summary changed: {name}')
reported = pd.read_csv(EVAL_DIR / 'validation_metrics.csv')
thresholds = pd.read_csv(EVAL_DIR / 'validation_thresholds.csv')
pixels = pd.read_csv(EVAL_DIR / 'pixel_and_failure_metrics.csv')
conditions = [f'{family}/{severity}' for family, severity in configuration['conditions']]
if len(conditions) != 10 or len(set(conditions)) != 10 or 'identity/none' not in conditions:
    raise RuntimeError('Expected ten unique validation conditions including identity.')
expected_units = {f'{family}_{severity}_{method}' for family, severity in configuration['conditions'] for method in METHODS}
if set(completion['unit_provenance']) != expected_units:
    raise RuntimeError('Completed unit provenance does not cover all forty variants.')
if thresholds.duplicated(['condition', 'method', 'dataset', 'db']).any():
    raise RuntimeError('Duplicate threshold records.')
reference_configuration = json.loads((REFERENCE_DIR / 'configuration.json').read_text(encoding='utf-8'))
reference_completion = json.loads((REFERENCE_DIR / 'completion.json').read_text(encoding='utf-8'))
if (file_digest(REFERENCE_DIR / 'configuration.json') != configuration['reference_evaluation_sha256']
        or file_digest(REFERENCE_DIR / 'completion.json') != configuration['reference_completion_sha256']
        or reference_completion['status'] != 'structure_validation_matching_complete'
        or reference_completion['configuration_sha256'] != configuration['reference_evaluation_sha256']):
    raise RuntimeError('Notebook-14 reference provenance changed; restore its frozen summaries.')
for name in ('configuration.json', 'validation_thresholds.csv', 'validation_metrics.csv', 'aggregate_operating_points.csv'):
    if file_digest(REFERENCE_DIR / name) != reference_completion['file_sha256'][name]:
        raise RuntimeError(f'Notebook-14 protected summary changed: {name}')
if (configuration['reference_reuse'] is not False or completion['training_performed'] is not False
        or completion['new_thresholds_calibrated'] is not False or completion['external_test_evaluated'] is not False):
    raise RuntimeError('Expected fresh notebook17 validation scores without training, calibration or external test.')
expected_conditions = {'identity/none'} | {f'{family}/{level}'
    for family in ('contact_gaps', 'directional_smear', 'jpeg_quantization') for level in ('level1', 'level2', 'level3')}
if set(conditions) != expected_conditions:
    raise RuntimeError('Unexpected alternative corruption condition/level grid.')
reference_thresholds = pd.read_csv(REFERENCE_DIR / 'validation_thresholds.csv')
reference_metrics = pd.read_csv(REFERENCE_DIR / 'validation_metrics.csv')
reference_aggregates = pd.read_csv(REFERENCE_DIR / 'aggregate_operating_points.csv')
reference_thresholds = reference_thresholds[
    reference_thresholds['condition'].eq('identity/none') & reference_thresholds['method'].eq('input')]
if reference_thresholds.duplicated(['dataset', 'db']).any() or len(reference_thresholds) != 6:
    raise RuntimeError('Expected six unique original-input reference thresholds.')
reference_fixed = {(row['dataset'], row['db']): row['original_fixed_threshold'] for row in reference_thresholds.to_dict('records')}
if set(configuration['fixed_thresholds']) != {f'{dataset}_{db}' for dataset, db in reference_fixed}:
    raise RuntimeError('Missing frozen threshold map entries.')
for row in thresholds.to_dict('records'):
    key = (row['dataset'], row['db'])
    value = reference_fixed.get(key)
    if (value is None or value != row['original_fixed_threshold']
            or value != configuration['fixed_thresholds'][f'{key[0]}_{key[1]}']
            or not pd.isna(row['condition_calibrated_threshold'])):
        raise RuntimeError('Notebook17 thresholds differ from saved original thresholds or contain new calibrations.')
units, extraction_frames, source_hash_rows = {}, [], []
canonical = None
for unit, provenance in sorted(completion['unit_provenance'].items()):
    directory = Path(provenance['directory'])
    marker_path = directory / 'completion.json'
    if file_digest(marker_path) != provenance['completion_sha256']:
        raise RuntimeError(f'Unit completion changed: {unit}')
    marker = json.loads(marker_path.read_text(encoding='utf-8'))
    if provenance['origin'] != 'notebook17':
        raise RuntimeError(f'Unexpected non-current-build unit provenance: {unit}')
    expected_config = completion['configuration_sha256']
    if marker['variant'] != unit or marker['configuration_sha256'] != expected_config:
        raise RuntimeError(f'Unit configuration/provenance mismatch: {unit}')
    for name in ('scores.csv', 'extraction.csv'):
        path = directory / name
        if file_digest(path) != marker['file_sha256'][name]:
            raise RuntimeError(f'Unit data changed: {path}')
        source_hash_rows.append({'unit': unit, 'file': name, 'sha256': marker['file_sha256'][name], 'path': str(path)})
    scored = pd.read_csv(directory / 'scores.csv')
    extraction = pd.read_csv(directory / 'extraction.csv')
    if scored['matchable'].dtype != bool or scored['duplicate_sensitivity_eligible'].dtype != bool:
        raise RuntimeError(f'Unexpected score booleans: {unit}')
    if scored['pair_id'].duplicated().any() or not set(scored['label']).issubset({0, 1}):
        raise RuntimeError(f'Invalid pair keys/labels: {unit}')
    valid_scores = scored.loc[scored['matchable'], 'score']
    if (not np.isfinite(valid_scores).all() or (valid_scores < 0).any()
            or not np.equal(valid_scores, np.floor(valid_scores)).all()
            or scored.loc[~scored['matchable'], 'score'].notna().any()):
        raise RuntimeError(f'Invalid score/failure encoding: {unit}')
    pair_columns = ['pair_id', 'dataset', 'db', 'label', 'probe_path', 'gallery_path',
                    'probe_identity', 'gallery_identity', 'duplicate_sensitivity_eligible']
    keys = scored[pair_columns].reset_index(drop=True)
    if canonical is None:
        canonical = keys
    elif not canonical.equals(keys):
        raise RuntimeError(f'Pair alignment differs across variants: {unit}')
    if len(scored) != completion['pairs_per_variant']:
        raise RuntimeError(f'Pair coverage differs: {unit}')
    condition, method = provenance['condition'], provenance['method']
    family, severity = condition.split('/')
    if method not in METHODS or unit != f'{family}_{severity}_{method}' or (condition, method) in units:
        raise RuntimeError(f'Invalid or duplicate method/condition provenance: {unit}')
    expected_probes = set(scored['probe_path'])
    if extraction['relative_path'].duplicated().any() or set(extraction['relative_path']) != expected_probes:
        raise RuntimeError(f'Incomplete extraction probe coverage: {unit}')
    if not extraction['condition'].eq(condition).all() or not extraction['method'].eq(method).all():
        raise RuntimeError(f'Extraction method/condition mismatch: {unit}')
    units[(condition, method)] = scored
    extraction_frames.append(extraction.assign(audit_condition=condition, audit_method=method))
databases = sorted(set(zip(canonical['dataset'], canonical['db'])))
if len(databases) != 6 or set(canonical['dataset']) != {'FVC2000', 'FVC2002'}:
    raise RuntimeError('Expected six source-only validation databases.')
print('Verified score variants:', len(units), '| Pairs per variant:', len(canonical))
print('No archives, images, train or test samples were opened.')
expected_threshold_keys = {(condition, method, dataset, db) for condition in conditions for method in METHODS for dataset, db in databases}
if set(zip(thresholds['condition'], thresholds['method'], thresholds['dataset'], thresholds['db'])) != expected_threshold_keys:
    raise RuntimeError('Incomplete frozen threshold table.')
if len(reported) != 40 * 6 * 2 or set(reported['threshold_policy']) != {'original_fixed'}:
    raise RuntimeError('Expected only fixed-threshold metrics for both sensitivity views.')
if set(reference_fixed) != set(databases):
    raise RuntimeError('Reference databases differ from notebook17.')


In [ ]:
helper_config_path = AUDIT12_DIR / 'configuration.json'
helper_completion = json.loads((AUDIT12_DIR / 'completion.json').read_text(encoding='utf-8'))
helper_config = json.loads(helper_config_path.read_text(encoding='utf-8'))
helper_path = AUDIT12_DIR / 'analysis_helpers.py.txt'
helper_bytes = helper_path.read_bytes()
helper_digest = hashlib.sha256(helper_bytes).hexdigest()
if (helper_completion['status'] != 'validation_tradeoff_audit_complete'
        or helper_completion['configuration_sha256'] != file_digest(helper_config_path)
        or helper_config['version'] != 'm1_validation_tradeoff_v1'
        or helper_config['replicates'] != REPLICATES or helper_config['seed'] != SEED
        or helper_digest != helper_completion['file_sha256']['analysis_helpers.py.txt']
        or helper_digest != helper_config['helper_sha256']):
    raise RuntimeError('Notebook-12 statistical helper provenance changed; preserve its completed analysis.')
helper_source = helper_bytes.decode('utf-8')
helper_filename = str(helper_path)
linecache.cache[helper_filename] = (len(helper_source), None, helper_source.splitlines(keepends=True), helper_filename)
exec(compile(helper_source, helper_filename, 'exec'), globals())
print('Verified and reused frozen notebook-12 rate and paired-bootstrap helpers.')


In [ ]:
finger_keys, bootstrap_weights = {}, {}
for dataset, db in databases:
    keys = sorted(canonical.loc[canonical['dataset'].eq(dataset) & canonical['db'].eq(db), 'probe_identity'].unique())
    if len(keys) < 2:
        raise RuntimeError('Need at least two probe-finger clusters per database.')
    finger_keys[(dataset, db)] = keys
    seed = int.from_bytes(hashlib.sha256(f'{SEED}|probe_cluster|{dataset}|{db}'.encode()).digest()[:8], 'big')
    bootstrap_weights[(dataset, db)] = np.random.default_rng(seed).multinomial(len(keys), np.full(len(keys), 1 / len(keys)), size=REPLICATES)
helper_source = '\n\n'.join(inspect.getsource(function) for function in (rates, percentile_interval, bootstrap_pair, interval_row))
analysis_config = {'version': 'alternative_corruption_tradeoff_v1', 'evaluation_configuration_sha256': completion['configuration_sha256'],
    'evaluation_completion_sha256': file_digest(EVAL_DIR / 'completion.json'), 'seed': SEED, 'replicates': REPLICATES,
    'policies': POLICIES, 'comparisons': COMPARISONS, 'methods': METHODS,
    'bootstrap': 'paired probe-finger clusters within database; equal database weights; same draws across conditions; fixed thresholds',
    'limitations': 'conditional on shared fixed galleries, calibration and selected checkpoints; pointwise intervals, not multiplicity corrected',
    'numpy_version': np.__version__, 'pandas_version': pd.__version__,
    'helper_sha256': hashlib.sha256(helper_source.encode()).hexdigest(),
    'notebook12_helper_completion_sha256': file_digest(AUDIT12_DIR / 'completion.json'),
    'consumed_unit_files_sha256': hashlib.sha256(json_bytes(source_hash_rows)).hexdigest(),
    'draw_sha256': {f'{dataset}_{db}': hashlib.sha256(weights.astype('<i8').tobytes()).hexdigest()
                    for (dataset, db), weights in bootstrap_weights.items()}}
if analysis_config['draw_sha256'] != helper_config['draw_sha256']:
    raise RuntimeError('Probe-finger bootstrap draws differ from frozen notebook12. Check validation identities and NumPy version; do not silently substitute draws.')
analysis_config['notebook14_reference_completion_sha256'] = file_digest(REFERENCE_DIR / 'completion.json')
analysis_config['alternative_protocol_sha256'] = configuration['alternative_protocol_sha256']
analysis_data = json_bytes(analysis_config)
analysis_sha = hashlib.sha256(analysis_data).hexdigest()
REPORT_DIR = PROJECT_ROOT / 'analyses' / 'alternative_corruption_tradeoff_v1' / analysis_sha[:20]
REPORT_DIR.mkdir(parents=True, exist_ok=True)
for name, data in {'configuration.json': analysis_data, 'analysis_helpers.py.txt': helper_source.encode()}.items():
    path = REPORT_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen analysis changed: {path}')
    if not path.exists():
        write_verified(path, data)
write_csv(REPORT_DIR / 'consumed_unit_files.csv', pd.DataFrame(source_hash_rows))
print('Analysis frozen:', analysis_sha)
print('Outputs:', REPORT_DIR)

In [ ]:
rate_rows, harms, threshold_lookup = [], [], {}
for condition in conditions:
    for method in METHODS:
        scored = units[(condition, method)]
        for policy in POLICIES:
            lookup = {}
            for (dataset, db), frame in scored.groupby(['dataset', 'db'], sort=True):
                saved = thresholds[thresholds['condition'].eq(condition) & thresholds['method'].eq(method)
                                   & thresholds['dataset'].eq(dataset) & thresholds['db'].eq(db)]
                if len(saved) != 1:
                    raise RuntimeError('Missing/duplicate frozen threshold record.')
                threshold = saved.iloc[0]['original_fixed_threshold' if policy == 'original_fixed' else 'condition_calibrated_threshold']
                if pd.isna(threshold):
                    raise RuntimeError(f'Unavailable frozen calibration: {condition}/{method}/{dataset}/{db}; do not invent a threshold.')
                lookup[(dataset, db)] = threshold
                for view, selected in (('all_impressions', frame), ('byte_distinct_sensitivity', frame[frame['duplicate_sensitivity_eligible']])):
                    values = rates(selected, threshold)
                    record = reported[reported['condition'].eq(condition) & reported['method'].eq(method)
                        & reported['dataset'].eq(dataset) & reported['db'].eq(db)
                        & reported['threshold_policy'].eq(policy) & reported['view'].eq(view)]
                    if len(record) != 1 or record.iloc[0]['threshold'] != threshold:
                        raise RuntimeError('Missing/duplicate or inconsistent notebook-17 metric row.')
                    for name, value in values.items():
                        previous = record.iloc[0][name]
                        if (value is None and not pd.isna(previous)) or (value is not None and (not np.isfinite(previous) or abs(previous - value) > 1e-10)):
                            raise RuntimeError(f'Recomputed notebook-17 metric mismatch: {condition}/{method}/{dataset}/{db}/{name}')
                    if policy == 'condition_calibrated' and view == 'all_impressions' and values['conditional_fmr'] is not None and values['conditional_fmr'] > 0.01 + 1e-12:
                        raise RuntimeError('Frozen calibration exceeds empirical conditional FMR 1%.')
                    rate_rows.append({'condition': condition, 'method': method, 'dataset': dataset, 'db': db,
                        'threshold_policy': policy, 'view': view, 'threshold': threshold, **values})
            threshold_lookup[(condition, method, policy)] = lookup
for condition in conditions:
    for policy in POLICIES:
        fixed = [threshold_lookup[(condition, method, policy)] for method in METHODS]
        if policy == 'original_fixed' and any(value != fixed[0] for value in fixed[1:]):
            raise RuntimeError('Original fixed thresholds differ across methods.')
        if policy == 'original_fixed' and fixed[0] != threshold_lookup[('identity/none', 'input', policy)]:
            raise RuntimeError('Original fixed thresholds differ across conditions.')
for baseline, method in COMPARISONS:
    left, right = units[('identity/none', baseline)], units[('identity/none', method)]
    for key in databases:
        dataset, db = key
        selected = left['dataset'].eq(dataset) & left['db'].eq(db) & left['label'].eq(1)
        threshold = threshold_lookup[('identity/none', 'input', 'original_fixed')][key]
        before = left.loc[selected, 'matchable'] & left.loc[selected, 'score'].ge(threshold)
        after = right.loc[selected, 'matchable'] & right.loc[selected, 'score'].ge(threshold)
        harms.append({'baseline_method': baseline, 'method': method, 'dataset': dataset, 'db': db,
            'genuine_attempts': int(selected.sum()), 'recoveries': int((~before & after).sum()), 'harms': int((before & ~after).sum())})
recomputed = pd.DataFrame(rate_rows)
write_csv(REPORT_DIR / 'recomputed_database_metrics.csv', recomputed)
write_csv(REPORT_DIR / 'original_image_harms.csv', pd.DataFrame(harms))
macro = recomputed.groupby(['condition', 'method', 'threshold_policy', 'view'], sort=True).agg(
    databases=('db', 'size'), attempt_frr=('attempt_frr', 'mean'), attempt_far=('attempt_far', 'mean'),
    conditional_fnmr=('conditional_fnmr', 'mean'), valid_fnmr_databases=('conditional_fnmr', 'count'),
    conditional_fmr=('conditional_fmr', 'mean'), valid_fmr_databases=('conditional_fmr', 'count')).reset_index()
if not macro['databases'].eq(6).all():
    raise RuntimeError('Macro summaries must cover all six databases.')
write_csv(REPORT_DIR / 'macro_operating_points.csv', macro)
print('All recomputed rates agree with frozen notebook-17 metrics.')
identity_checks = []
for row in recomputed[recomputed['condition'].eq('identity/none')].to_dict('records'):
    previous = reference_metrics[reference_metrics['condition'].eq('identity/none')
        & reference_metrics['method'].eq(row['method']) & reference_metrics['dataset'].eq(row['dataset'])
        & reference_metrics['db'].eq(row['db']) & reference_metrics['threshold_policy'].eq('original_fixed')
        & reference_metrics['view'].eq(row['view'])]
    if len(previous) != 1:
        raise RuntimeError('Missing/duplicate original-image reference metric.')
    for name in ('threshold', 'genuine_attempts', 'impostor_attempts', 'unmatchable_genuine',
                 'unmatchable_impostor', 'attempt_frr', 'attempt_far', 'conditional_fmr', 'conditional_fnmr'):
        left, right = row[name], previous.iloc[0][name]
        if not ((pd.isna(left) and pd.isna(right)) or (np.isfinite(left) and np.isfinite(right) and abs(left - right) <= 1e-10)):
            raise RuntimeError(f'Original-image identity metric differs from notebook14: {row["method"]}/{row["dataset"]}/{row["db"]}/{name}')
    identity_checks.append({key: row[key] for key in ('method', 'dataset', 'db', 'view')})
write_csv(REPORT_DIR / 'identity_reference_check.csv', pd.DataFrame(identity_checks).assign(matches_notebook14=True))
print('Original-image identity metrics agree with notebook14, including duplicate sensitivity.')


In [ ]:
interval_rows, replicate_results = [], {}
for condition in conditions:
    for policy in POLICIES:
        for baseline, method in COMPARISONS:
            values = bootstrap_pair(units[(condition, baseline)], units[(condition, method)],
                threshold_lookup[(condition, baseline, policy)], threshold_lookup[(condition, method, policy)])
            replicate_results[(condition, policy, baseline, method)] = values
            interval_rows.append(interval_row(condition, policy, baseline, method, values))
    print('Paired probe-finger bootstrap:', condition, flush=True)
synthetic_conditions = [condition for condition in conditions if condition != 'identity/none']
scopes = {'synthetic_nine_equal_weight': synthetic_conditions,
          **{f'{family}_three_levels': [condition for condition in conditions if condition.startswith(family + '/')]
             for family in ('contact_gaps', 'directional_smear', 'jpeg_quantization')}}
for scope, members in scopes.items():
    if len(members) != (9 if scope == 'synthetic_nine_equal_weight' else 3):
        raise RuntimeError('Unexpected aggregate condition coverage.')
    for policy in POLICIES:
        for baseline, method in COMPARISONS:
            pieces = [replicate_results[(condition, policy, baseline, method)] for condition in members]
            values = {rate: {'delta': float(np.mean([piece[rate]['delta'] for piece in pieces])),
                      'replicates': np.mean([piece[rate]['replicates'] for piece in pieces], axis=0)} for rate in ('frr', 'far')}
            interval_rows.append(interval_row(scope, policy, baseline, method, values))
intervals = pd.DataFrame(interval_rows)
write_csv(REPORT_DIR / 'paired_probe_finger_intervals.csv', intervals)
aggregate_rows = []
for scope, members in scopes.items():
    subset = macro[macro['condition'].isin(members)]
    grouped = subset.groupby(['method', 'threshold_policy', 'view'], sort=True).agg(
        conditions=('condition', 'size'), attempt_frr=('attempt_frr', 'mean'), attempt_far=('attempt_far', 'mean'),
        conditional_fnmr=('conditional_fnmr', 'mean'), conditional_fmr=('conditional_fmr', 'mean')).reset_index()
    if not grouped['conditions'].eq(len(members)).all():
        raise RuntimeError('Aggregate operating points have incomplete conditions.')
    aggregate_rows.append(grouped.assign(scope=scope))
aggregates = pd.concat(aggregate_rows, ignore_index=True)
write_csv(REPORT_DIR / 'aggregate_operating_points.csv', aggregates)
print('Deltas = method minus baseline, in percentage points; negative FRR is better, positive FAR is worse.')
print(intervals[intervals['scope'].eq('synthetic_nine_equal_weight')].to_string(index=False))

In [ ]:
all_extraction = pd.concat(extraction_frames, ignore_index=True)
pixel_summary = all_extraction.groupby(['audit_condition', 'audit_method'], sort=True).agg(
    images=('relative_path', 'size'), float_input_mae=('float_input_mae', 'mean'),
    float_output_mae=('float_output_mae', 'mean'), png_input_mae=('png_input_mae', 'mean'),
    png_output_mae=('png_output_mae', 'mean'), changed_pixel_fraction=('changed_pixel_fraction', 'mean'),
    identical_image_fraction=('png_identical_to_input', 'mean')).reset_index()
write_csv(REPORT_DIR / 'image_weighted_pixel_summary.csv', pixel_summary)
print('\nOperating points at saved original thresholds only; no calibration. Read FRR and FAR together.')
print(aggregates[aggregates['scope'].eq('synthetic_nine_equal_weight') & aggregates['view'].eq('all_impressions')].to_string(index=False))
print('\nOriginal-image recoveries/harms at the SAME original thresholds:')
print(pd.DataFrame(harms).groupby(['baseline_method', 'method'])[['recoveries', 'harms']].sum().to_string())
print('\nFamily-specific paired changes (pointwise, conditional intervals; not significance tests):')
print(intervals[intervals['scope'].str.endswith('three_levels') & intervals['threshold_policy'].eq('original_fixed')].to_string(index=False))
context_rows = []
for domain, frame in (
    ('training_formula_validation', reference_aggregates[reference_aggregates['threshold_policy'].eq('original_fixed')]),
    ('alternative_corruption_validation', aggregates[aggregates['view'].eq('all_impressions')])):
    for scope, group in frame.groupby('scope', sort=True):
        baseline = group[group['method'].eq('input')]
        if len(group) != len(METHODS) or set(group['method']) != set(METHODS) or len(baseline) != 1:
            raise RuntimeError('Incomplete contextual operating points.')
        prefix = 'macro_' if domain == 'training_formula_validation' else ''
        for row in group.to_dict('records'):
            context_rows.append({'domain': domain, 'scope': scope, 'method': row['method'],
                'attempt_frr': row[prefix + 'attempt_frr'], 'attempt_far': row[prefix + 'attempt_far'],
                'delta_frr_vs_input_pp': 100 * (row[prefix + 'attempt_frr'] - baseline.iloc[0][prefix + 'attempt_frr']),
                'delta_far_vs_input_pp': 100 * (row[prefix + 'attempt_far'] - baseline.iloc[0][prefix + 'attempt_far']),
                'interpretation': 'context only; unequal difficulty; shared validation identities; not pooled or independent'})
context_points = pd.DataFrame(context_rows)
write_csv(REPORT_DIR / 'development_context_point_estimates.csv', context_points)
print('\nContext only, NOT a matched-severity generalization contrast:')
print(context_points[context_points['scope'].isin(['synthetic_nine', 'synthetic_nine_equal_weight'])].to_string(index=False))
files = ['configuration.json', 'analysis_helpers.py.txt', 'consumed_unit_files.csv', 'recomputed_database_metrics.csv',
    'original_image_harms.csv', 'macro_operating_points.csv', 'paired_probe_finger_intervals.csv',
    'aggregate_operating_points.csv', 'image_weighted_pixel_summary.csv', 'identity_reference_check.csv', 'development_context_point_estimates.csv']
marker = {'status': 'alternative_corruption_audit_complete', 'configuration_sha256': analysis_sha,
    'file_sha256': {name: file_digest(REPORT_DIR / name) for name in files},
    'evaluation_completion_sha256': file_digest(EVAL_DIR / 'completion.json'),
    'training_performed': False, 'new_thresholds_calibrated': False, 'test_evaluated': False,
    'independent_test_improvement_verified': False,
    'limitations': analysis_config['limitations']}
write_verified(REPORT_DIR / 'completion.json', json_bytes(marker))
print('Do not pool protocols or treat unpaired severity levels as equivalent. Keep test sealed; do not tune from these intervals.')
print('\nMatched Sobel versus L1: primary fixed-threshold identity and family intervals:')
selected = intervals['baseline_method'].eq('l1_control') & intervals['method'].eq('l1_sobel') & intervals['threshold_policy'].eq('original_fixed')
print(intervals[selected & (intervals['scope'].eq('identity/none') | intervals['scope'].str.endswith('three_levels') | intervals['scope'].eq('synthetic_nine_equal_weight'))].to_string(index=False))
print('ALTERNATIVE CORRUPTION AUDIT COMPLETE:', REPORT_DIR)
print('Exploratory conditional intervals only. No training, threshold changes, held-out test or real-moisture claims.')
